In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def computeELO(elos, ranks, k):
  n = len(elos)
  if n <=1:
    return elos

  scores = (n - ranks) / (n-1)

  diff = (elos.reshape(-1,1) - elos.reshape(1,-1)) /400
  expected = 1/(1+10**(-diff))
  np.fill_diagonal(expected, np.nan)
  expected = np.nanmean(expected, axis=1)

  return elos + k * (scores - expected)

In [ ]:
df = pd.read_csv('IndyCar_dataset_v14.csv')

ritmo = 5
DNFwindow = 20
baseELO = 1500.0
k = 64
tk = 48
ek = 32

ordered_cols = [
    "DriverName",
    "DriverID",
    "Rookie",
    "DRFAvg",
    "DTAvg",
    "DTTAvg",
    "DNFRate",
    "TDNFRate",
    "DriverElo",
    "DriverTElo",
    "DriverTTElo",
    "DriverRitmo",
    "PositionStart",
    "TeamName",
    "TeamID",
    "TRP",
    "TTP",
    "TeamDNFRate",
    "TeamElo",
    "TeamTElo",
    "TeamRitmo",
    "CarEngine",
    "EngineID",
    "EngineElo",
    "EngineTElo",
    "EngineTTElo",
    "EventName",
    "Track",
    "TrackID",
    "EventTrackType",
    "EventTrackTypeID",
    "EventDate",
    "EventDateFormatted",
    "EventID",
    "Era",
    "EraID",
    "Status",
    "StatusID",
    "FieldSize",
    "PositionFinish",
    "NormalizedPositionFinish"
]

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID"]).reset_index(drop=True)

In [ ]:
df["NormalizedPositionFinish"] = (
    (df["PositionFinish"]-1)/(df["FieldSize"]-1)
)

In [ ]:
df["DRFAvg"] = (
    df.groupby("DriverID")["NormalizedPositionFinish"].transform(lambda s: s.shift(1).rolling(ritmo, min_periods=1).mean())
)

In [ ]:
df["DTAvg"] = (
    df.groupby(["DriverID", "TrackID"])["NormalizedPositionFinish"].transform(lambda s: s.shift(1).expanding().mean())
)

In [ ]:
df['DTTAvg'] = (
    df.groupby(["DriverID", "EventTrackTypeID"])["NormalizedPositionFinish"].transform(lambda s: s.shift(1).expanding().mean())
)

In [ ]:
df["DNFRate"] = (
    df.groupby("DriverID")["StatusID"].transform((lambda s: s.shift(1).rolling(DNFwindow, min_periods=1).mean()))
)

In [ ]:
df["TDNFRate"] = (
    df.groupby(["DriverID", "TrackID"])["StatusID"].transform((lambda s: s.shift(1).expanding().mean()))
)

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID"]).reset_index(drop=True)

driverElo ={}
df["DriverElo"] = np.nan

for eventID, event in df.groupby("EventID", sort=False):
  indx = event.index
  drivers = event["DriverID"].values
  ranks = event["PositionFinish"].values.astype(float)

  preElo = np.array([driverElo.get(d, baseELO) for d in drivers], dtype=float)
  df.loc[indx, "DriverElo"] = preElo

  postElo = computeELO(preElo, ranks, k)
  for d, new in zip(drivers, postElo):
    driverElo[d] = float(new)


driverTrackElo = {}
df["DriverTElo"] = np.nan

for eventID, event in df.groupby("EventID", sort=False):
    indx = event.index
    drivers = event["DriverID"].values
    tracks = event["TrackID"].values
    ranks = event["PositionFinish"].values.astype(float)

    preElo = np.array([driverTrackElo.get((d, t), baseELO) for d, t in zip(drivers, tracks)], dtype=float)
    df.loc[indx, "DriverTElo"] = preElo

    postElo = computeELO(preElo, ranks, k)
    for d, t, new in zip(drivers, tracks, postElo):
        driverTrackElo[(d, t)] = float(new)

driverTrackTypeElo = {}
df["DriverTTElo"] = np.nan

for eventID, event in df.groupby("EventID", sort=False):
    indx = event.index
    drivers = event["DriverID"].values
    tracktypes = event["EventTrackTypeID"].values
    ranks = event["PositionFinish"].values.astype(float)

    preElo = np.array([driverTrackTypeElo.get((d, tt), baseELO) for d, tt in zip(drivers, tracktypes)], dtype=float)
    df.loc[indx, "DriverTTElo"] = preElo

    postElo = computeELO(preElo, ranks, k)
    for d, tt, new in zip(drivers, tracktypes, postElo):
        driverTrackTypeElo[(d, tt)] = float(new)


In [ ]:
df["DriverRitmo"] = (
    df.groupby("DriverID")["NormalizedPositionFinish"].transform(lambda s: s.shift(1).ewm(ritmo, min_periods=1).mean())
)

In [ ]:
from types import TracebackType
df = df.drop(columns=["TRP", "TTP", "TeamDNFRate", "TeamElo",  "TeamTElo", "TeamRitmo"])
df["EventDate"] = pd.to_datetime(df["EventDate"])

teamdf = (
    df.groupby(["EventID", "TeamID"], as_index=False)
    .agg(
        teamPerf=("NormalizedPositionFinish", "mean"),
        EventTrackTypeID=("EventTrackTypeID", "first"),
        TrackID=("TrackID", "first"),
        EventDate=("EventDate", "first"),
        teamDNF=("StatusID", "max")
    ).sort_values(["EventDate", "EventID", "TeamID"]).reset_index(drop=True)
)

In [ ]:
teamdf = teamdf.sort_values(["EventDate", "EventID", "TeamID"]).reset_index(drop=True)

In [ ]:
teamdf["TRP"] = (
    teamdf.groupby("TeamID")["teamPerf"].transform(lambda s: s.shift(1).rolling(ritmo, min_periods=1).mean())
)

In [ ]:
teamdf["TTP"] = (
    teamdf.groupby(["TeamID", "EventTrackTypeID"])["teamPerf"].transform(lambda s: s.shift(1).expanding().mean())
)

In [ ]:
teamdf["TeamDNFRate"] = (
     teamdf.groupby("TeamID")["teamDNF"].transform(lambda s: s.shift(1).rolling(DNFwindow, min_periods=1).mean())
)

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "TeamID"]).reset_index(drop=True)

teamElo ={}
teamdf["TeamElo"] = np.nan

for eventID, event in teamdf.groupby("EventID", sort=False):
  indx = event.index
  teams = event["TeamID"].values
  ranks = event["teamPerf"].rank(method="first").values.astype(float)

  preElo = np.array([teamElo.get(t, baseELO) for t in teams], dtype=float)
  teamdf.loc[indx, "TeamElo"] = preElo

  postElo = computeELO(preElo, ranks, tk)
  for t, new in zip(teams, postElo):
    teamElo[t] = float(new)

teamTrackElo = {}
teamdf["TeamTElo"] = np.nan

for eventID, event in teamdf.groupby("EventID", sort=False):
    indx = event.index
    teams = event["TeamID"].values
    tracks = event["TrackID"].values
    ranks = event["teamPerf"].rank(method="first").values.astype(float)

    preElo = np.array([teamTrackElo.get((t, tr), baseELO) for t, tr in zip(teams, tracks)], dtype=float)
    teamdf.loc[indx, "TeamTElo"] = preElo

    postElo = computeELO(preElo, ranks, tk)
    for t, tr, new in zip(teams, tracks, postElo):
        teamTrackElo[(t, tr)] = float(new)

In [ ]:
teamdf["TeamRitmo"] = (
    teamdf.groupby("TeamID")["teamPerf"].transform(lambda s: s.shift(1).ewm(ritmo, min_periods=1).mean())
)

In [ ]:
df = df.merge(
    teamdf[["EventID", "TeamID", "TRP", "TTP", "TeamDNFRate", "TeamElo", "TeamTElo", "TeamRitmo"]],
    on=["EventID", "TeamID"],
    how="left"
)

df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "TeamID", "DriverID"]).reset_index(drop=True)
df = df[ordered_cols]

In [ ]:
df = df.drop(columns=["EngineElo", "EngineTElo", "EngineTTElo"])
df["EventDate"] = pd.to_datetime(df["EventDate"])

enginedf = (
    df.groupby(["EventID", "EngineID"], as_index=False)
    .agg(
        enginePerf=("NormalizedPositionFinish", "mean"),
        EventTrackTypeID=("EventTrackTypeID", "first"),
        TrackID=("TrackID", "first"),
        EventDate=("EventDate", "first"),
    ).sort_values(["EventDate", "EventID", "EngineID"]).reset_index(drop=True)
)

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "EngineID"]).reset_index(drop=True)

engineElo ={}
enginedf["EngineElo"] = np.nan

for eventID, event in enginedf.groupby("EventID", sort=False):
  indx = event.index
  engines = event["EngineID"].values
  ranks = event["enginePerf"].rank(method="first").values.astype(float)

  preElo = np.array([engineElo.get(e, baseELO) for e in engines], dtype=float)
  enginedf.loc[indx, "EngineElo"] = preElo

  postElo = computeELO(preElo, ranks, ek)
  for e, new in zip(engines, postElo):
    engineElo[e] = float(new)

engineTrackElo = {}
enginedf["EngineTElo"] = np.nan

for eventID, event in enginedf.groupby("EventID", sort=False):
    indx = event.index
    engines = event["EngineID"].values
    tracks = event["TrackID"].values
    ranks = event["enginePerf"].rank(method="first").values.astype(float)

    preElo = np.array([engineTrackElo.get((e, tr), baseELO) for e, tr in zip(engines, tracks)], dtype=float)
    enginedf.loc[indx, "EngineTElo"] = preElo

    postElo = computeELO(preElo, ranks, ek)
    for e, tr, new in zip(engines, tracks, postElo):
        engineTrackElo[(e, tr)] = float(new)

engineTrackTypeElo = {}
enginedf["EngineTTElo"] = np.nan

for eventID, event in enginedf.groupby("EventID", sort=False):
    indx = event.index
    engines = event["EngineID"].values
    tracktypes = event["EventTrackTypeID"].values
    ranks = event["enginePerf"].rank(method="first").values.astype(float)

    preElo = np.array([engineTrackTypeElo.get((e, tt), baseELO) for e, tt in zip(engines, tracktypes)], dtype=float)
    enginedf.loc[indx, "EngineTTElo"] = preElo

    postElo = computeELO(preElo, ranks, ek)
    for e, tt, new in zip(engines, tracktypes, postElo):
        engineTrackTypeElo[(e, tt)] = float(new)

In [ ]:
df = df.merge(
    enginedf[["EventID", "EngineID", "EngineElo", "EngineTElo", "EngineTTElo"]],
    on=["EventID", "EngineID"],
    how="left"
)

df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "EngineID", "DriverID"]).reset_index(drop=True)
df = df[ordered_cols]

In [ ]:
df.to_csv("IndyCar_dataset_14.csv")